# ASC Phase 2 — Training

**Mục tiêu:** Huấn luyện mô hình `roberta-base` cho Aspect Sentiment Classification (0=neg, 1=neu, 2=pos).

**Pipeline:**
1. Load file 2M mẫu đã sample sẵn từ `ASC_PHASE_2/sample` → `sampled_df`
2. Kết hợp với gold train; loại leakage từ gold test
3. Format aspect-marker → fine-tune `roberta-base`
4. Đo metrics trên gold test: accuracy, macro/weighted f1/precision/recall


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

import os

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR        = "/content/drive/MyDrive/outputs_electronics_cleaning"
ASC_SAMPLE_DIR  = f"{BASE_DIR}/ASC_PHASE_2/sampled"   # thư mục chứa sampled_2M.parquet
GOLD_DIR        = f"{BASE_DIR}/gold_with_sentiments"
SCRIPT_DIR      = BASE_DIR

MODEL_OUTPUT_DIR = f"{BASE_DIR}/ASC_PHASE_2/model"
REPORT_DIR       = f"{BASE_DIR}/ASC_PHASE_2/reports"

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR,       exist_ok=True)

SEED = 123

# ── Training hyperparams ──────────────────────────────────────────────────────
MODEL_NAME          = "roberta-base"
MAX_LENGTH          = 128    # 128 đủ cho >95% cặp (sentence, aspect); giảm ~30% thời gian
NUM_EPOCHS          = 3      # 2M mẫu → 3 epoch đủ (~3.7h T4); 5 epoch rủi ro Colab timeout
LEARNING_RATE       = 2e-5
WEIGHT_DECAY        = 0.01
WARMUP_RATIO        = 0.06
EARLY_STOP_PATIENCE = 5      # 5 × 4000 steps = 1/3 epoch mới dừng → tránh dừng sớm
EVAL_STEPS          = 4000   # ~4 lần eval/epoch
FP16                = True
BF16                = True   # ưu tiên BF16 trên A100/H100

print(f"BASE_DIR         : {BASE_DIR}")
print(f"ASC_SAMPLE_DIR   : {ASC_SAMPLE_DIR}")
print(f"GOLD_DIR         : {GOLD_DIR}")
print(f"MODEL_OUTPUT_DIR : {MODEL_OUTPUT_DIR}")
print(f"MAX_LENGTH       : {MAX_LENGTH}")
print(f"NUM_EPOCHS       : {NUM_EPOCHS}  (~{NUM_EPOCHS * 75:.0f} min T4 estimate)")
print(f"EVAL_STEPS       : {EVAL_STEPS}  (early_stop_patience={EARLY_STOP_PATIENCE})")


BASE_DIR         : /content/drive/MyDrive/outputs_electronics_cleaning
ASC_SAMPLE_DIR   : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_2/sampled
GOLD_DIR         : /content/drive/MyDrive/outputs_electronics_cleaning/gold_with_sentiments
MODEL_OUTPUT_DIR : /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_2/model
MAX_LENGTH       : 128
NUM_EPOCHS       : 3  (~225 min T4 estimate)
EVAL_STEPS       : 4000  (early_stop_patience=5)


In [3]:
import sys, re, ast
import numpy as np
import pandas as pd
import torch
import gc
from glob import glob
from tqdm.auto import tqdm

import pyarrow.parquet as pq

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score,
    classification_report, confusion_matrix,
)

# Import mark_aspect / clean_text từ Phase 1 script
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
from analyze_aspect_sentiment import clean_text, mark_aspect

LABEL_NAMES = ["negative", "neutral", "positive"]

print("Imports OK")
print(f"Device: {'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'cpu'}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Imports OK
Device: cuda (Tesla T4)


In [4]:
# ── Bước 1: Load file 2M mẫu đã sample sẵn ──────────────────────────────────

paths = sorted(glob(f"{ASC_SAMPLE_DIR}/*.parquet"))
if not paths:
    raise FileNotFoundError(f"Không tìm thấy file .parquet trong: {ASC_SAMPLE_DIR}")

print(f"Tìm thấy {len(paths)} file parquet:")
for p in paths:
    print(f"  {p}")

parts = []
for path in tqdm(paths, desc="Loading"):
    df = pq.read_table(path).to_pandas()

    # Hỗ trợ cả 2 format:
    #   (a) đã flatten: cột "aspect" + "label"
    #   (b) chưa flatten: cột "aspects" (list) + "sentiments" (list)
    if "aspects" in df.columns and "aspect" not in df.columns:
        df = df.explode(["aspects", "sentiments"]).reset_index(drop=True)
        df = df.rename(columns={"aspects": "aspect"})
        df["label"] = df["sentiments"].apply(lambda x: int(np.argmax(x)))
        df = df.drop(columns=["sentiments"], errors="ignore")

    parts.append(df[["sentence_id", "sentence_text", "aspect", "label", "category_name"]])

sampled_df = pd.concat(parts, ignore_index=True)
print(f"\nTổng: {len(sampled_df):,} cặp")
print(f"  neg={(sampled_df.label==0).sum():,}  "
      f"neu={(sampled_df.label==1).sum():,}  "
      f"pos={(sampled_df.label==2).sum():,}")


Tìm thấy 1 file parquet:
  /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_2/sampled/sampled_2M.parquet


Loading:   0%|          | 0/1 [00:00<?, ?it/s]


Tổng: 2,000,000 cặp
  neg=966,628  neu=66,745  pos=966,627


In [5]:
# ── Bước 3: Load gold train + test, flatten, parse labels ────────────────────

def parse_list_col(series):
    if series.dtype == object and isinstance(series.iloc[0], str):
        return series.apply(ast.literal_eval)
    return series


def flatten_gold(path):
    df = pd.read_csv(path)
    df["aspects"]    = parse_list_col(df["aspects"])
    df["sentiments"] = parse_list_col(df["sentiments"])
    df = df.explode(["aspects", "sentiments"]).reset_index(drop=True)
    df = df.rename(columns={"aspects": "aspect"})
    df["label"] = df["sentiments"].apply(lambda x: int(np.argmax(x)))
    return df[["sentence_id", "sentence_text", "aspect", "label", "category_name"]]


gold_train_df = flatten_gold(f"{GOLD_DIR}/gold_train.csv")
gold_test_df  = flatten_gold(f"{GOLD_DIR}/gold_test.csv")

print(f"Gold train : {len(gold_train_df):,} cặp")
print(f"Gold test  : {len(gold_test_df):,} cặp")
print(f"\nGold test distribution:")
print(gold_test_df["label"].value_counts().sort_index().rename({0:"neg",1:"neu",2:"pos"}))

Gold train : 4,443 cặp
Gold test  : 1,128 cặp

Gold test distribution:
label
neg    602
neu     30
pos    496
Name: count, dtype: int64


In [6]:
# ── Bước 4: Loại leakage → kết hợp train ────────────────────────────────────

test_pairs = set(zip(gold_test_df["sentence_id"], gold_test_df["aspect"]))
print(f"Test pairs cần loại: {len(test_pairs):,}")


def remove_leakage(df, test_pairs):
    keys = list(zip(df["sentence_id"], df["aspect"]))
    mask = [k not in test_pairs for k in keys]
    removed = len(df) - sum(mask)
    if removed:
        print(f"  Loại bỏ {removed:,} dòng leakage")
    return df[mask].reset_index(drop=True)


sampled_df    = remove_leakage(sampled_df,    test_pairs)
gold_train_df = remove_leakage(gold_train_df, test_pairs)

train_df = (
    pd.concat([sampled_df, gold_train_df], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

print(f"\nTrain sau khi kết hợp & loại leakage: {len(train_df):,}")
print("Label distribution:")
for lbl, name in enumerate(LABEL_NAMES):
    cnt = (train_df["label"] == lbl).sum()
    print(f"  {name:8s}: {cnt:>10,}  ({cnt/len(train_df)*100:.1f}%)")

del sampled_df, gold_train_df; gc.collect()

Test pairs cần loại: 943
  Loại bỏ 439 dòng leakage

Train sau khi kết hợp & loại leakage: 2,004,004
Label distribution:
  negative:    968,864  (48.3%)
  neutral :     66,872  (3.3%)
  positive:    968,268  (48.3%)


99

In [7]:
# ── Bước 5: Tokenizer + Dataset ──────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
num_added = tokenizer.add_special_tokens(
    {"additional_special_tokens": ["[ASP]", "[/ASP]"]}
)
print(f"Special tokens added: {num_added}  |  vocab size: {len(tokenizer)}")


class ASCDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=192):
        # Vectorized: build tất cả texts trước
        self.texts = [
            mark_aspect(str(r.sentence_text), str(r.aspect))
            for r in df.itertuples(index=False)
        ]
        self.labels     = df["label"].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        return {**enc, "labels": self.labels[idx]}


train_dataset = ASCDataset(train_df,    tokenizer, MAX_LENGTH)
test_dataset  = ASCDataset(gold_test_df, tokenizer, MAX_LENGTH)

print(f"Train dataset: {len(train_dataset):,}")
print(f"Test  dataset: {len(test_dataset):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Special tokens added: 2  |  vocab size: 50267
Train dataset: 2,004,004
Test  dataset: 1,128


In [8]:
# ── Bước 6: Model + compute_metrics ─────────────────────────────────────────

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2},
)
model.resize_token_embeddings(len(tokenizer))  # do thêm [ASP] [/ASP]

# Tự chọn batch size & precision theo VRAM / compute capability
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap     = torch.cuda.get_device_capability(0)
    print(f"GPU : {torch.cuda.get_device_name(0)}  ({vram_gb:.1f} GB)  compute={cap}")
    TRAIN_BATCH  = 128 if vram_gb >= 40 else 32 if vram_gb >= 15 else 16
    USE_BF16     = BF16 and cap[0] >= 8      # BF16: Ampere+ (sm80+)
    USE_FP16     = FP16 and (not USE_BF16)
    USE_TF32     = cap[0] >= 8               # tf32 + torch_compile: Ampere+ only
else:
    TRAIN_BATCH  = 16
    USE_BF16     = False
    USE_FP16     = False
    USE_TF32     = False
    print("WARNING: không có GPU")

GRAD_ACCUM = max(1, 128 // TRAIN_BATCH)     # effective batch = 128
print(f"Batch={TRAIN_BATCH}  grad_accum={GRAD_ACCUM}  effective={TRAIN_BATCH*GRAD_ACCUM}")
print(f"Precision : {'BF16' if USE_BF16 else 'FP16' if USE_FP16 else 'FP32'}")
print(f"TF32/Compile: {USE_TF32}")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy"          : float(accuracy_score(labels, preds)),
        "f1_macro"          : float(f1_score(labels, preds, average="macro",    zero_division=0)),
        "f1_weighted"       : float(f1_score(labels, preds, average="weighted", zero_division=0)),
        "precision_macro"   : float(precision_score(labels, preds, average="macro",    zero_division=0)),
        "recall_macro"      : float(recall_score(labels,  preds, average="macro",    zero_division=0)),
        "precision_weighted": float(precision_score(labels, preds, average="weighted", zero_division=0)),
        "recall_weighted"   : float(recall_score(labels,  preds, average="weighted", zero_division=0)),
        "f1_neg"            : float(f1_score(labels, preds, labels=[0], average="micro", zero_division=0)),
        "f1_neu"            : float(f1_score(labels, preds, labels=[1], average="micro", zero_division=0)),
        "f1_pos"            : float(f1_score(labels, preds, labels=[2], average="micro", zero_division=0)),
    }


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and

GPU : Tesla T4  (15.6 GB)  compute=(7, 5)
Batch=32  grad_accum=4  effective=128
Precision : FP16
TF32/Compile: False


In [9]:

# ── Bước 7: TrainingArguments + Trainer ──────────────────────────────────────
import time
from transformers import TrainerCallback

class PrintTimeCallback(TrainerCallback):
    """In elapsed time, ETA và loss sau mỗi eval checkpoint (không lẫn với tqdm)."""
    def on_train_begin(self, args, state, control, **kwargs):
        self._t0 = time.time()
        print("Training started...")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        elapsed  = time.time() - self._t0
        step     = state.global_step
        max_step = state.max_steps if state.max_steps and state.max_steps > 0 else 1
        pct      = step / max_step
        eta      = (elapsed / pct - elapsed) if pct > 0 else 0
        f1_str   = f"{metrics.get('eval_f1_macro', metrics.get('eval_f1', 0)):.4f}" if metrics else "?"
        print(
            f"  [Eval] step={step:>6}/{max_step}  "
            f"f1_macro={f1_str}  "
            f"elapsed={elapsed/60:.1f}min  ETA={eta/60:.1f}min"
        )

# Tính warmup_steps thủ công (warmup_ratio deprecated từ v5.2)
total_steps   = (len(train_dataset) // (TRAIN_BATCH * GRAD_ACCUM)) * NUM_EPOCHS
warmup_steps  = int(WARMUP_RATIO * total_steps)
print(f"total_steps={total_steps:,}  warmup_steps={warmup_steps:,}")

# Số worker an toàn cho Colab (T4 có ít CPU/RAM; 2 là ổn định nhất)
DL_WORKERS = 2

training_args = TrainingArguments(
    output_dir                  = MODEL_OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = TRAIN_BATCH,
    per_device_eval_batch_size  = TRAIN_BATCH * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    warmup_steps                = warmup_steps,
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = EVAL_STEPS,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",
    greater_is_better           = True,
    save_total_limit            = 2,
    fp16                        = USE_FP16,
    bf16                        = USE_BF16,
    tf32                        = USE_TF32,          # Ampere+ only (A100, RTX 30xx+)
    group_by_length             = True,              # giảm padding → tăng tốc ~10%
    optim                       = "adamw_torch_fused",
    torch_compile               = USE_TF32,          # Ampere+ only
    logging_steps               = 200,
    dataloader_num_workers      = DL_WORKERS,
    dataloader_prefetch_factor  = 2 if DL_WORKERS > 0 else None,
    report_to                   = "none",
    seed                        = SEED,
)

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = test_dataset,
    processing_class = tokenizer,       # transformers v5: thay tokenizer= → processing_class=
    data_collator    = collator,
    compute_metrics  = compute_metrics,
    callbacks        = [
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
        PrintTimeCallback(),
    ],
)

print("Trainer ready.")


total_steps=46,968  warmup_steps=2,818
Trainer ready.


In [10]:

# ── Bước 8: Train (có hỗ trợ resume từ checkpoint) ───────────────────────────
import glob as _glob

# Tìm checkpoint mới nhất trong MODEL_OUTPUT_DIR
_ckpts = sorted(
    _glob.glob(f"{MODEL_OUTPUT_DIR}/checkpoint-*"),
    key=lambda p: int(p.rsplit("-", 1)[-1])   # sort theo số step
)
resume_from = _ckpts[-1] if _ckpts else None

if resume_from:
    print(f">>> Resume từ checkpoint: {resume_from}")
    print(f"    (Tìm thấy {len(_ckpts)} checkpoint: {[c.split('/')[-1] for c in _ckpts]})")
else:
    print(">>> Không có checkpoint — train từ đầu")

train_result = trainer.train(resume_from_checkpoint=resume_from)
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print(f"\nTrain done.")
print(f"  Steps   : {train_result.global_step}")
print(f"  Loss    : {train_result.training_loss:.4f}")
print(f"  Runtime : {train_result.metrics.get('train_runtime', 0)/60:.1f} min")


>>> Resume từ checkpoint: /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_2/model/checkpoint-28000
    (Tìm thấy 3 checkpoint: ['checkpoint-20000', 'checkpoint-24000', 'checkpoint-28000'])


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training started...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro,Precision Weighted,Recall Weighted,F1 Neg,F1 Neu,F1 Pos
32000,0.075361,0.697342,0.903369,0.667934,0.894896,0.851886,0.649862,0.899911,0.903369,0.922570,0.176471,0.904762
36000,0.074693,0.639986,0.907801,0.670904,0.899279,0.854822,0.652868,0.904245,0.907801,0.927512,0.176471,0.908730
40000,0.073967,0.688429,0.901596,0.665148,0.893495,0.800874,0.648755,0.895272,0.901596,0.924606,0.171429,0.899408
44000,0.075784,0.840784,0.901596,0.681842,0.894318,0.823215,0.659194,0.896773,0.901596,0.922314,0.222222,0.900990


  [Eval] step= 32000/46971  f1_macro=0.6679  elapsed=19.1min  ETA=8.9min


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Train done.
  Steps   : 46971
  Loss    : 0.0286
  Runtime : 91.2 min


In [11]:
# ── Bước 9: Evaluate trên gold test ─────────────────────────────────────────

pred_output = trainer.predict(test_dataset)
preds  = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

acc      = accuracy_score(labels, preds)
f1_mac   = f1_score(labels, preds, average="macro",    zero_division=0)
f1_wei   = f1_score(labels, preds, average="weighted", zero_division=0)
prec_mac = precision_score(labels, preds, average="macro",    zero_division=0)
rec_mac  = recall_score(labels,  preds, average="macro",    zero_division=0)
prec_wei = precision_score(labels, preds, average="weighted", zero_division=0)
rec_wei  = recall_score(labels,  preds, average="weighted", zero_division=0)

report_str = classification_report(labels, preds, target_names=LABEL_NAMES, digits=4)
cm         = confusion_matrix(labels, preds)
cm_df      = pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES)

print("=" * 62)
print("  ASC Phase 2 — Gold Test Benchmark")
print("=" * 62)
print(f"  Accuracy            : {acc:.4f}")
print(f"  Macro Precision     : {prec_mac:.4f}")
print(f"  Macro Recall        : {rec_mac:.4f}")
print(f"  Macro F1            : {f1_mac:.4f}")
print(f"  Weighted F1         : {f1_wei:.4f}")
print(f"  Weighted Precision  : {prec_wei:.4f}")
print(f"  Weighted Recall     : {rec_wei:.4f}")
print(f"\nPer-class:\n{report_str}")
print(f"Confusion matrix (rows=true, cols=pred):\n{cm_df.to_string()}")


  ASC Phase 2 — Gold Test Benchmark
  Accuracy            : 0.9051
  Macro Precision     : 0.8422
  Macro Recall        : 0.6711
  Macro F1            : 0.6998
  Weighted F1         : 0.8985
  Weighted Precision  : 0.9012
  Weighted Recall     : 0.9051

Per-class:
              precision    recall  f1-score   support

    negative     0.9073    0.9435    0.9251       602
     neutral     0.7143    0.1667    0.2703        30
    positive     0.9051    0.9032    0.9041       496

    accuracy                         0.9051      1128
   macro avg     0.8422    0.6711    0.6998      1128
weighted avg     0.9012    0.9051    0.8985      1128

Confusion matrix (rows=true, cols=pred):
          negative  neutral  positive
negative       568        1        33
neutral         11        5        14
positive        47        1       448


In [12]:
# ── Bước 9: Lưu báo cáo ra Google Drive ─────────────────────────────────────

steps_per_epoch = len(train_dataset) // (TRAIN_BATCH * GRAD_ACCUM)
trained_epochs  = train_result.global_step // max(steps_per_epoch, 1)

report_lines = [
    "=" * 62,
    "  ASC Phase 2 — Gold Test Benchmark Report",
    "=" * 62,
    "",
    f"  Model            : {MODEL_NAME}",
    f"  Train samples    : {len(train_df):,}",
    f"  Test  samples    : {len(gold_test_df):,}",
    f"  Seed             : {SEED}",
    f"  Max length       : {MAX_LENGTH}",
    f"  Learning rate    : {LEARNING_RATE}",
    f"  Epochs (trained) : {trained_epochs}",
    f"  Best model       : {MODEL_OUTPUT_DIR}",
    "",
    "[ Overall Metrics ]",
    f"  Accuracy            : {acc:.4f}",
    f"  Macro Precision     : {prec_mac:.4f}",
    f"  Macro Recall        : {rec_mac:.4f}",
    f"  Macro F1            : {f1_mac:.4f}",
    f"  Weighted F1         : {f1_wei:.4f}",
    f"  Weighted Precision  : {prec_wei:.4f}",
    f"  Weighted Recall     : {rec_wei:.4f}",
    "",
    "[ Per-class Report ]",
    report_str,
    "[ Confusion Matrix ]  rows=true  cols=pred",
    cm_df.to_string(),
    "",
    "=" * 62,
]

report_path = f"{REPORT_DIR}/asc_phase2_benchmark.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"Báo cáo đã lưu: {report_path}")


Báo cáo đã lưu: /content/drive/MyDrive/outputs_electronics_cleaning/ASC_PHASE_2/reports/asc_phase2_benchmark.txt
